# KVBridge — cloud calibration and evaluation

Run this notebook on a **cloud GPU**, not your low-VRAM local computer. Start with the three small **stand-in** models and five GSM8K test questions. The notebook calibrates both model hand-offs, proves a cold fallback, compares all four assignments with a no-cross-step-cache baseline, and exports measured results.

**Before starting:** commit and push your latest project code, including this notebook, to GitHub. The cloud clone cannot see uncommitted local files. Alternatively, upload a complete source checkout and set REPO_DIR to it.

For Colab, upload this notebook using **File → Upload notebook**, then choose a GPU under **Runtime → Change runtime type**. A regular GPU-backed Jupyter session also works. Run cells in order. Model downloads stay disabled until you set ENABLE_MODEL_DOWNLOADS to True in the next cell.

Colab GPU availability and runtime duration are not guaranteed, and its VM files can disappear when a session ends. Download your results before disconnecting. See the [official Colab FAQ](https://research.google.com/colaboratory/faq.html). This notebook does not rent compute or purchase credits for you.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/henry-goldhub/multiagent-kvcache.git"
REPO_REF = ""  # Optional exact pushed commit SHA; blank uses the remote default branch.
REPO_DIR = (Path("/content") if Path("/content").exists() else Path.cwd()) / "kvbridge-cloud-source"

PROFILE = "standin"  # "standin" first; "full" is the optional named-model experiment.
ENABLE_MODEL_DOWNLOADS = False  # Set True when ready to download models and GSM8K.
ENABLE_FULL_PROFILE = False  # Must also be True for PROFILE="full".
HF_LOGIN = False  # Usually unnecessary for public models. Enable only if access requires it.
TRANSFORMERS_VERSION = "5.16.1"  # Version tested locally; record changes as a new experiment.

SEED = 42
NUM_EXAMPLES = 5  # Increase to 200 only after the small real-model run works.
MAX_NEW_TOKENS = 96
FIT_EXAMPLES = 16
VALIDATION_EXAMPLES = 8
PROBE_TOKENS = 8
KL_THRESHOLD = 0.15
RIDGE_LAMBDA = 0.001
DOWNLOAD_ZIP = False  # Colab: toggle True in the export cell when ready.

assert PROFILE in {"standin", "full"}
assert all(value > 0 for value in (
    NUM_EXAMPLES, MAX_NEW_TOKENS, FIT_EXAMPLES, VALIDATION_EXAMPLES, PROBE_TOKENS
))
assert PROFILE != "full" or ENABLE_FULL_PROFILE, (
    "Full-profile loading is disabled. First finish the stand-in run; "
    "then explicitly enable the full profile in a fresh runtime."
)

## 1. Get the exact source and install

An existing checkout is reused without pulling, resetting, or overwriting it. To switch revisions, choose a new REPO_DIR or manage Git yourself. Keep tokens out of REPO_URL. A private repository needs a separately authenticated checkout.

The install uses the runtime's existing PyTorch when it satisfies the package requirements. Do not upgrade CUDA/PyTorch blindly. If the runtime reports that a restart is needed, restart the kernel and rerun from the top before importing KVBridge.

In [ ]:
import os
import subprocess
import sys

new_checkout = not REPO_DIR.exists()
if new_checkout:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    if REPO_REF:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True
        )
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("REPO_DIR exists but is not a Git checkout. Choose a new folder.")

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if REPO_REF and not new_checkout:
    requested_commit = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", f"{REPO_REF}^{{commit}}"], text=True
    ).strip()
    if SOURCE_COMMIT != requested_commit:
        raise RuntimeError("Existing checkout differs from REPO_REF. Use a new REPO_DIR.")
if subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
).strip():
    raise RuntimeError("Checkout has uncommitted changes. Commit them before a measured run.")

for relative in (
    "kvbridge/calibration.py", "kvbridge/quality.py", "kvbridge/compat.py",
    "examples/quickstart.py", "tests/test_cache_formats.py",
):
    if not (REPO_DIR / relative).is_file():
        raise RuntimeError(f"Missing {relative}. Push the implementation and clone the right commit.")

os.chdir(REPO_DIR)
extras = "dev,eval,full" if PROFILE == "full" else "dev,eval"
requirements = [
    f"{REPO_DIR}[{extras}]",
    f"transformers=={TRANSFORMERS_VERSION}",
    "accelerate>=1.0",
]
if PROFILE == "full":
    requirements.append("sentencepiece>=0.2")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", requirements[0], *requirements[1:]],
    check=True,
)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "examples"))
print("Source commit:", SOURCE_COMMIT)

## 2. Check hardware and run the offline tests first

The stand-in loader uses float32 and eager attention for a conservative compatibility run. The full loader uses 4-bit weights and one common compute dtype. These choices are recorded and affect comparisons.

Calibration loads **two models at a time** and releases them between pairs. Mixed evaluation currently holds **all three models on one GPU**; it does not implement sequential model swapping. The full profile can need substantial GPU and CPU RAM despite 4-bit weights. No GPU-memory size is guaranteed to fit. If it runs out of memory, stop and use a larger runtime or stay with stand-ins—do not claim the full run completed.

In [ ]:
import gc
import importlib.metadata
import json
import platform
import random
import shutil
import time
import uuid
from dataclasses import asdict
from datetime import datetime, timezone

import torch
import transformers
from IPython.display import FileLink, Markdown, display
from quickstart import ASSIGNMENTS, MODEL_PROFILES
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from kvbridge import Pipeline, evaluate, report_to_markdown
from kvbridge.adapter import RidgeKVAdapter
from kvbridge.calibration import calibrate_model_pair
from kvbridge.evaluation import save_report
from kvbridge.models import ModelBundle
from kvbridge.prompts import PromptState

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Enable a GPU runtime and reconnect before continuing.")
if transformers.__version__ != TRANSFORMERS_VERSION:
    raise RuntimeError("Transformers was already imported at another version. Restart the kernel.")

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
RUN_DTYPE = (
    torch.bfloat16 if PROFILE == "full" and torch.cuda.is_bf16_supported()
    else torch.float16 if PROFILE == "full"
    else torch.float32
)
gpu = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f"GPU: {gpu.name}; free {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
print("Python:", platform.python_version(), "Torch:", torch.__version__)
print("Transformers:", transformers.__version__, "Compute dtype:", RUN_DTYPE)
print(f"Free disk: {shutil.disk_usage(REPO_DIR).free / 2**30:.1f} GiB")

In [ ]:
# These tests and the synthetic quickstart do not download model weights or datasets.
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-m", "not integration and not full_models"],
    cwd=REPO_DIR, check=True,
)
subprocess.run(
    [sys.executable, "-m", "ruff", "check", "kvbridge", "tests", "examples"],
    cwd=REPO_DIR, check=True,
)
subprocess.run(
    [sys.executable, "examples/quickstart.py", "--profile", "synthetic",
     "--num-examples", "2", "--max-new-tokens", "2"],
    cwd=REPO_DIR, check=True,
)

## 3. Enable the cloud experiment

The next cells download pretrained weights and the GSM8K train/test splits. Edit ENABLE_MODEL_DOWNLOADS above and rerun that settings cell before continuing. Watch your provider's usage/billing and disconnect when done.

If a repository requires authentication, enable HF_LOGIN. The hidden prompt calls Hugging Face's [official login API](https://huggingface.co/docs/huggingface_hub/package_reference/authentication) without adding a Git credential. Never paste a token into a code cell or include it in exported output.

In [ ]:
if not ENABLE_MODEL_DOWNLOADS:
    raise RuntimeError(
        "Download gate is closed. Set ENABLE_MODEL_DOWNLOADS=True, rerun settings, then continue here."
    )
if HF_LOGIN:
    from getpass import getpass

    from huggingface_hub import login

    token = getpass("Hugging Face read token (hidden): ")
    try:
        login(token=token, add_to_git_credential=False)
    finally:
        del token

## 4. Start an isolated run and freeze the data/model identities

Every execution creates a fresh output directory. Adapter filenames use aliases such as model_1, so **never share adapter directories across profiles, model revisions, or runs**.

The record includes the source commit, package versions, GPU, requested and resolved model revisions, complete settings, and the exact GSM8K row indices. Training and test subsets are selected separately using the same fixed seed. Gold test answers are used only for evaluation, never for calibration.

In [ ]:
# These small orchestration helpers are also smoke-tested offline with synthetic models.
def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")


def choose_rows(dataset, count, seed):
    if count <= 0 or count > len(dataset):
        raise ValueError("Requested subset must be nonempty and fit within the dataset.")
    indices = list(range(len(dataset)))
    random.Random(seed).shuffle(indices)
    indices = indices[:count]
    return indices, [dict(dataset[index]) for index in indices]


def release_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def calibrate_handoff(loader, source_name, target_name, fit_rows, validation_rows,
                      adapter_dir, ridge_lambda, threshold, probe_tokens):
    source_bundle = target_bundle = None
    try:
        source_bundle = loader(source_name)
        target_bundle = loader(target_name)
        # Use a long probe and explicitly check its target-token length. The library
        # truncates to probe_tokens, so a short "\nContinue:" probe would be insufficient.
        probe_text = (
            " Continue solving the arithmetic problem using the quantities "
            "and relationships supplied in the question."
        ) * max(8, probe_tokens)
        encoded = target_bundle.tokenizer(
            probe_text, return_tensors="pt", add_special_tokens=False
        )
        if encoded.input_ids.shape[-1] < probe_tokens:
            raise ValueError("Probe text does not contain enough target tokens.")
        adapter = RidgeKVAdapter(adapter_dir, ridge_lambda=ridge_lambda)
        projection, quality = calibrate_model_pair(
            adapter,
            source_bundle,
            target_bundle,
            [PromptState.from_question(row["question"]).text for row in fit_rows],
            [PromptState.from_question(row["question"]).text for row in validation_rows],
            probe_text=probe_text,
            probe_tokens=probe_tokens,
            threshold=threshold,
        )
        return {
            "metadata": asdict(projection.metadata),
            "quality": asdict(quality),
            "probe_tokens": probe_tokens,
            "probe_text": probe_text,
        }
    finally:
        del source_bundle, target_bundle
        release_gpu()


def prove_missing_artifact_fallback(models, question, config, output_dir):
    config = {
        **config,
        "max_new_tokens": 1,
        "adapter": {
            "type": "ridge",
            "artifact_dir": str(Path(output_dir) / f"intentionally-missing-{uuid.uuid4().hex}"),
        },
    }
    result, logs = Pipeline(models, config).run(question, ["model_1", "model_2", "model_3"])
    for step in logs["steps"][1:]:
        assert step["cold_prefill"] and not step["adapter_accepted"]
        assert step["fallback_reason"] == "missing_calibration"
    payload = {"test": "forced missing-artifact fallback; excluded from measurements",
               "result": result, "logs": logs}
    write_json(Path(output_dir) / "forced_fallback.json", payload)
    return payload


def evaluate_with_checkpoints(models, examples, assignments, config, output_dir):
    output_dir = Path(output_dir)
    report = {"seed": config["seed"], "cache_policy": config["cache_policy"], "settings": {}}
    wall_seconds = {}
    # Save after every assignment so a later failure does not erase earlier results.
    def checkpoint(complete, error_type=None):
        save_report(report, output_dir / "report.json")
        (output_dir / "report.md").write_text(report_to_markdown(report), encoding="utf-8")
        write_json(output_dir / "evaluation_status.json", {
            "complete": complete,
            "completed_assignments": len(report["settings"]),
            "expected_assignments": len(assignments),
            "combined_baseline_and_cached_wall_seconds": wall_seconds,
            "error_type": error_type,
        })

    checkpoint(False)
    try:
        for assignment in assignments:
            name = " -> ".join(assignment)
            print("Evaluating:", name, flush=True)
            started = time.perf_counter()
            partial = evaluate(Pipeline(models, config), examples, [assignment], config)
            wall_seconds[name] = time.perf_counter() - started
            report["settings"].update(partial["settings"])
            checkpoint(False)
        checkpoint(True)
    except Exception as error:
        checkpoint(False, type(error).__name__)
        raise
    return report


def archive_run(run_dir):
    run_dir = Path(run_dir).resolve()
    # Only package this run's files, never the repository, HF cache, or credentials.
    return Path(shutil.make_archive(
        str(run_dir.parent / f"{run_dir.name}-export"), "zip", root_dir=run_dir
    ))

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN_DIR = REPO_DIR / "results" / "cloud" / f"{PROFILE}-{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
ADAPTER_DIR = RUN_DIR / "adapters"
ADAPTER_DIR.mkdir()
MODEL_IDS = MODEL_PROFILES[PROFILE]
REQUESTED_REVISIONS = {alias: "main" for alias in MODEL_IDS}

environment = {
    "source_commit": SOURCE_COMMIT,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_bytes": total_bytes,
    "gpu_free_bytes_at_start": free_bytes,
    "compute_dtype": str(RUN_DTYPE),
    "attention_implementation": "eager",
    "placement": "all evaluation models resident on cuda:0",
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("torch", "transformers", "safetensors", "datasets", "accelerate")
    },
}
if PROFILE == "full":
    environment["packages"]["bitsandbytes"] = importlib.metadata.version("bitsandbytes")
write_json(RUN_DIR / "environment.json", environment)
CONFIG = {
    "cache_policy": "cross_model",
    "seed": SEED,
    "device": "auto",  # Respect the loader; never .to() a quantized model afterward.
    "max_new_tokens": MAX_NEW_TOKENS,
    "step_assignments": ASSIGNMENTS,
    "adapter": {"type": "ridge", "artifact_dir": str(ADAPTER_DIR),
                "ridge_lambda": RIDGE_LAMBDA},
    "degradation": {"metric": "mean_kl", "threshold": KL_THRESHOLD,
                    "probe_tokens": PROBE_TOKENS},
}
write_json(RUN_DIR / "config.json", {
    "profile": PROFILE,
    "pipeline": CONFIG,
    "num_examples": NUM_EXAMPLES,
    "fit_examples": FIT_EXAMPLES,
    "validation_examples": VALIDATION_EXAMPLES,
    "model_ids": MODEL_IDS,
    "requested_revisions": REQUESTED_REVISIONS,
})
print("Run directory:", RUN_DIR)

In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi

api = HfApi()
RESOLVED_REVISIONS = {
    alias: api.model_info(model_id, revision=REQUESTED_REVISIONS[alias]).sha
    for alias, model_id in MODEL_IDS.items()
}
write_json(RUN_DIR / "model_revisions.json", RESOLVED_REVISIONS)

train_data = load_dataset("openai/gsm8k", "main", split="train")
test_data = load_dataset("openai/gsm8k", "main", split="test")
train_indices, training_rows = choose_rows(
    train_data, FIT_EXAMPLES + VALIDATION_EXAMPLES, SEED
)
test_indices, eval_rows = choose_rows(test_data, NUM_EXAMPLES, SEED)
fit_rows = training_rows[:FIT_EXAMPLES]
validation_rows = training_rows[FIT_EXAMPLES:]
write_json(RUN_DIR / "dataset_subsets.json", {
    "dataset": "openai/gsm8k",
    "configuration": "main",
    "seed": SEED,
    "train_fingerprint": getattr(train_data, "_fingerprint", None),
    "test_fingerprint": getattr(test_data, "_fingerprint", None),
    "fit_train_indices": train_indices[:FIT_EXAMPLES],
    "validation_train_indices": train_indices[FIT_EXAMPLES:],
    "test_indices": test_indices,
    "fit_rows": fit_rows,
    "validation_rows": validation_rows,
    "eval_rows": eval_rows,
})
print(f"Fit: {len(fit_rows)}; held-out validation: {len(validation_rows)}; test: {len(eval_rows)}")

## 5. Define the cloud loader

The notebook uses the repository's model IDs but fixes model and tokenizer revisions to the same resolved commit. Loading is explicit on cuda:0 rather than silently offloading models to CPU and presenting those times as GPU-only results. Remote repository code execution is disabled.

This cell defines the loader; weights are loaded by calibration/evaluation below. Use the same dtype, quantization, and attention implementation throughout one run.

In [ ]:
def load_cloud_bundle(alias):
    model_id = MODEL_IDS[alias]
    revision = RESOLVED_REVISIONS[alias]
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, trust_remote_code=False
    )
    kwargs = {
        "revision": revision,
        "device_map": {"": 0},
        "dtype": RUN_DTYPE,
        "attn_implementation": "eager",
        "trust_remote_code": False,
    }
    if PROFILE == "full":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=RUN_DTYPE,
        )
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    return ModelBundle(model=model, tokenizer=tokenizer, name=alias)

## 6. Calibrate both hand-offs

Each pair fits independent affine ridge maps for keys and values at every target layer, after deterministic layer/head/sequence alignment. It then measures held-out mean KL against cold-target logits over exactly eight teacher-forced probe positions (or PROBE_TOKENS if changed).

**A rejected pair is a valid outcome.** Its metadata is saved and the runtime should fall back. Do not raise the threshold merely to manufacture cache hits. Unexpected exceptions stop this cell; export partial results and investigate before evaluation.

Current calibration uses question/extract prefixes as a proxy, not generated stage-2/3 histories. A low probe KL is not proof of downstream GSM8K accuracy; the evaluation is still necessary. In particular, these probes are not eight greedily generated answer tokens.

In [ ]:
calibration_results = {}
for source_name, target_name in (("model_1", "model_2"), ("model_2", "model_3")):
    pair_name = f"{source_name} -> {target_name}"
    print("Calibrating:", pair_name, flush=True)
    write_json(RUN_DIR / "calibration_status.json", {
        "complete": False, "current_pair": pair_name,
        "completed_pairs": list(calibration_results),
    })
    started = time.perf_counter()
    try:
        result = calibrate_handoff(
            load_cloud_bundle, source_name, target_name,
            fit_rows, validation_rows, ADAPTER_DIR,
            RIDGE_LAMBDA, KL_THRESHOLD, PROBE_TOKENS,
        )
    except Exception as error:
        write_json(RUN_DIR / "calibration_status.json", {
            "complete": False, "failed_pair": pair_name,
            "completed_pairs": list(calibration_results),
            "error_type": type(error).__name__,
        })
        raise
    result["wall_seconds"] = time.perf_counter() - started
    calibration_results[pair_name] = result
    write_json(RUN_DIR / "calibration_results.json", calibration_results)
    print(pair_name, result["quality"], flush=True)

write_json(RUN_DIR / "calibration_status.json", {
    "complete": True, "completed_pairs": list(calibration_results),
})

## 7. Load the evaluation models and prove fallback

The forced-fallback smoke test deliberately points at a missing adapter directory. It runs real pretrained models, asserts both switched stages cold-prefill with the precise reason, and saves their logs. It is separate from the measured run and never modifies the calibrated artifacts.

If any loading or forward call fails, stop here. A failing adapter forward must not be reported as a successful fallback.

In [ ]:
if not json.loads((RUN_DIR / "calibration_status.json").read_text())["complete"]:
    raise RuntimeError("Finish calibration before evaluation.")
# Release models from a previous execution of this cell before replacing the dictionary.
globals().pop("models", None)
release_gpu()
models = {}
try:
    for alias in MODEL_IDS:
        print("Loading for evaluation:", alias, flush=True)
        models[alias] = load_cloud_bundle(alias)
except Exception:
    models.clear()
    release_gpu()
    raise

fallback_proof = prove_missing_artifact_fallback(
    models, eval_rows[0]["question"], CONFIG, RUN_DIR
)
print("Verified missing-calibration cold fallback at both hand-offs.")
print([step["fallback_reason"] for step in fallback_proof["logs"]["steps"]])

## 8. Warm up and evaluate all four assignments

Warm-up runs are excluded from metrics. Each measured assignment uses the same questions, seed, and decoding limit for both policies. The baseline disables cross-step reuse while retaining normal intra-generation KV caching.

JSON contains baseline/cached accuracy, latency distributions, hit rates, and adapter rates; Markdown shows the selected-policy summary with speedups. Component timing is **not full end-to-end latency**: it excludes loading/tokenization and may have GPU adapter-boundary accounting limitations. Combined wall time is recorded separately, not mislabeled as a speedup. Five examples are a smoke test, not a reliable benchmark.

Reports are checkpointed after each assignment. Only a status file with complete=true means all four finished.

In [ ]:
warmup_rows = [{"question": "A basket has two apples and receives one more. How many apples?",
                "answer": "#### 3"}]
warmup_config = {**CONFIG, "max_new_tokens": 2}
for assignment in ASSIGNMENTS:
    for policy in ("disabled", "cross_model"):
        Pipeline(models, {**warmup_config, "cache_policy": policy}).run(
            warmup_rows[0]["question"], assignment
        )
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

report = evaluate_with_checkpoints(
    models, eval_rows, ASSIGNMENTS, CONFIG, RUN_DIR
)
write_json(RUN_DIR / "gpu_peak_memory.json", {
    "max_allocated_bytes": torch.cuda.max_memory_allocated(),
    "max_reserved_bytes": torch.cuda.max_memory_reserved(),
})
display(Markdown(report_to_markdown(report)))

## 9. Inspect actual decisions and preserve an example trace

This additional, unmeasured mixed run saves the final answer and per-stage cache decisions. Look for same-model cache hits in the report and cross-model accepted hits **or explained fallbacks** in the trace. The artificial missing-artifact proof must not be used as evidence that a calibrated adapter improved quality.

In [ ]:
answer, logs = Pipeline(models, CONFIG).run(
    eval_rows[0]["question"], ["model_1", "model_2", "model_3"]
)
write_json(RUN_DIR / "mixed_example_trace.json", {
    "question": eval_rows[0]["question"],
    "gold_answer": eval_rows[0]["answer"],
    "prediction": answer,
    "logs": logs,
    "included_in_measured_report": False,
})
for index, step in enumerate(logs["steps"], start=1):
    print(
        f"Stage {index}: {step['cache_decision']}; "
        f"hit_tokens={step['cache_hit_tokens']}; reason={step['fallback_reason']}"
    )

## 10. Export before disconnecting — also works after a partial failure

The ZIP contains this run's reports, config, data selection, hardware/package records, traces, status files, and pair artifacts. It excludes model weights, the Hugging Face cache, and credentials. Read evaluation_status.json and calibration_status.json before treating an export as complete.

For Colab, set DOWNLOAD_ZIP=True below. In regular Jupyter, use the displayed file link or the file browser. You can rerun this cell after any earlier failure once RUN_DIR exists. Do not publish synthetic or incomplete results as pretrained benchmarks.

In [ ]:
# Set True here to trigger the Colab download dialog.
DOWNLOAD_ZIP = False

archive = archive_run(RUN_DIR)
print("Export:", archive)
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        display(FileLink(os.path.relpath(archive, Path.cwd())))
    else:
        files.download(str(archive))
else:
    display(FileLink(os.path.relpath(archive, Path.cwd())))

## Finish / larger run

1. Inspect the saved errors, calibration decisions, and accuracy before changing settings.
2. Preserve the five-example result. Then start a **new run directory** with NUM_EXAMPLES=200 if compute permits. Reuse the source/model revisions to make comparisons meaningful.
3. The optional full-model experiment requires PROFILE="full" and ENABLE_FULL_PROFILE=True, a fresh runtime, enough GPU/CPU memory, and recalibration. Never copy stand-in adapters into it.
4. Add only measured, correctly labeled results to README.md. A table containing rejected adapters and no cross-model speedup is an honest research result.
5. Download your ZIP, then disconnect/release the cloud runtime in your provider's UI to stop consuming resources.

Troubleshooting: authentication errors require model access, not a pasted public token; missing source files usually mean the newest commit was not pushed; out-of-memory requires smaller workloads or a larger runtime; cache API/model exceptions require a code fix and a new source commit, not fabricated results.

In [ ]:
# Release notebook-held model references. This does NOT stop provider billing.
if "models" in globals():
    del models
release_gpu()
print("Model references released. Download results, then disconnect the cloud runtime.")